In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

Note: you may need to restart the kernel to use updated packages.
  Using cached qiskit-aer-0.15.1.tar.gz (6.6 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build qiskit-aer
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Building wheel for qiskit-aer (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [113 lines of output]
      C:\Users\berni\AppData\Local\Temp\pip-build-env-mkxpzrt2\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: Apache Software License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      
      
      --------------------------------------------------------------------------------

Note: you may need to restart the kernel to use updated packages.


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [3]:
# Generate a random bit using quantum measurement
def quantum_random_bit():

    qc = QuantumCircuit(1, 1)

    qc.h(0)

    qc.measure(0, 0)

    simulator = BasicSimulator()

    compiled = transpile(qc, simulator)

    result = simulator.run(compiled, shots=1).result()

    counts = result.get_counts()

    return int(list(counts.keys())[0])

In [4]:
# Generate a sequence of random bits
def generate_random_bits(n):

    bits = []

    for i in range(n):
        bits.append(quantum_random_bit())

    return bits

In [5]:
# Encode a bit using either the standard or diagonal basis
def prepare_qubit(bit, basis):

    qc = QuantumCircuit(1, 1)

    # Encode bit value
    if bit == 1:
        qc.x(0)

    # Convert to diagonal basis
    if basis == 1:
        qc.h(0)

    return qc

In [6]:
# Measure a qubit using the chosen basis
def measure_qubit(qc, basis):

    # Convert diagonal basis back to standard basis
    if basis == 1:
        qc.h(0)

    qc.measure(0, 0)

    simulator = BasicSimulator()

    compiled = transpile(qc, simulator)

    result = simulator.run(compiled, shots=1).result()

    counts = result.get_counts()

    return int(list(counts.keys())[0])

In [7]:
n = 100

# =========================
# Alice's part (Sender)
# =========================

## Alice generates random bits and bases
alice_bits = generate_random_bits(n)
alice_bases = generate_random_bits(n)

qubits = []

for i in range(n):

    qubit = prepare_qubit(
        alice_bits[i],
        alice_bases[i]
    )

    qubits.append(qubit)


In [8]:
# =========================
# Eve's part (Attacker)
# =========================

## Eve randomly chooses bases
eve_bases = generate_random_bits(n)

eve_bits = []

resent_qubits = []

for i in range(n):

    # Eve measures Alice's qubit
    eve_bit = measure_qubit(
        qubits[i],
        eve_bases[i]
    )

    eve_bits.append(eve_bit)

    # Eve resends a new qubit to bob
    resent_qubit = prepare_qubit(
        eve_bit,
        eve_bases[i]
    )

    resent_qubits.append(resent_qubit)

In [9]:
# =========================
# Bob's part (Receiver)
# =========================

## Bob randomly choose bases and measure the received qubits
bob_bases = generate_random_bits(n)

bob_bits = []

for i in range(n):

    measured_bit = measure_qubit(
        resent_qubits[i],
        bob_bases[i]
    )

    bob_bits.append(measured_bit)

In [10]:
# ======================================
# Public discussion and attack detection
# ======================================

matching_positions = []

for i in range(n):

    if alice_bases[i] == bob_bases[i]:
        matching_positions.append(i)

alice_sifted_key = []
bob_sifted_key = []

# Generate sifted keys using matching positions
for i in matching_positions:

    alice_sifted_key.append(alice_bits[i])
    bob_sifted_key.append(bob_bits[i])

# Alice and Bob publicly compare part of the sifted key
test_size = len(alice_sifted_key) // 2

alice_test_bits = alice_sifted_key[:test_size]
bob_test_bits = bob_sifted_key[:test_size]

errors = 0

# Count the number of mismatched bits
for i in range(len(alice_sifted_key)):

    if alice_sifted_key[i] != bob_sifted_key[i]:
        errors = errors + 1

# Calculate the error rate
error_rate = errors / len(alice_sifted_key)

# Threshold used to detect possible eavesdropping
threshold = 0.10

if error_rate > threshold:
    result = "Attack detected. Key rejected."
else:
    result = "No attack detected. Key accepted."

# Remaining bits form the final shared key
final_alice_key = alice_sifted_key[test_size:]
final_bob_key = bob_sifted_key[test_size:]

print("Alice bits: ", alice_bits)
print("Alice bases:", alice_bases)

print("Eve bases:  ", eve_bases)
print("Eve bits:   ", eve_bits)

print("Bob bases:  ", bob_bases)
print("Bob bits:   ", bob_bits)

print("Matching positions:", matching_positions)

print("Alice sifted key:", alice_sifted_key)
print("Bob sifted key:  ", bob_sifted_key)

print("Test size:", test_size)
print("Errors:", errors)
print("Error rate:", round(error_rate, 2))
print(result)

print("Final Alice key:", final_alice_key)
print("Final Bob key:  ", final_bob_key)

Alice bits:  [1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1]
Alice bases: [1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0]
Eve bases:   [1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1]
Eve bits:    [1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1,

In [ ]:
# The results show that Eve intercepts Alice's qubits, measures them, and resends new qubits to Bob.
# Eve's measurements may disturb the qubit states if she chooses the wrong basis, which can introduce errors into the sifted key.
# Because the basis choices are random, some runs may detect an attack while others may not, especially when the number of transmitted qubits is small.
# 100 qubits were used in this simulation so that the effects of eavesdropping and the resulting error rate are easier to observe.
# This follows the BB84 protocol from the lecture slides, where Alice and Bob compare matching-basis bits and use the error rate to detect possible eavesdropping.